**Reading from silver layer**

In [0]:
one_df = spark.read.table("clinical_trials_drug_data.silver.silver_data")
patient_df = spark.read.table("clinical_trials_drug_data.silver.patient_silver_data")
patient_df.createOrReplaceTempView("patient_df")
drug_df = spark.read.table("clinical_trials_drug_data.silver.drug_silver_data")

In [0]:
patient_df1 = spark.sql("select * from patient_df where age_group in ('Young', 'Adult')")

In [0]:
patient_df1.show()

In [0]:
patient_df1.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.gold.dim_patient_gold_data")

In [0]:
drug_df.show()

In [0]:
drug_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.gold.dim_drug_gold_data")

In [0]:
display(one_df.count())

In [0]:
one_df.display()
one_df.createOrReplaceTempView("one_df")

In [0]:
success_df = spark.sql("select * from one_df where outcome IN ('Success', 'Ongoing')")
success_df.display()

In [0]:
success_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.gold.fact_success_data")

In [0]:
faild_df = spark.sql("select * from one_df where outcome = 'Failure'")

In [0]:
faild_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.gold.fact_faild_data")

In [0]:
faild_df.display()

**pipeline_logs_Gold**

In [0]:
from datetime import datetime

start_time = datetime.now()

trial_df = spark.read.table("clinical_trials_drug_data.silver.silver_data")

In [0]:
from pyspark.sql import Row

log_data = [Row(
    pipeline_name="clinical_pipeline_gold",
    layer_name="Gold",
    table_name="trial_info",
    record_count=trial_df.count(),
    status="SUCCESS",
    start_time=str(start_time),
    end_time=str(datetime.now()),
    error_message=""
)]

spark.createDataFrame(log_data) \
    .write.format("delta") \
    .mode("append") \
    .saveAsTable("clinical_trials_drug_data.gold.pipeline_logs_Gold")

In [0]:
df = spark.read.table("clinical_trials_drug_data.gold.pipeline_logs_Gold")
df.display()